Grain Boundary Analysis (for use with grain boundary masks or any other comparison between two mask types)

Inputs: 
- ROOT_PATH: Containing all raw data 
- FINAL_PLOTS_SAVE_PATH: For saving data

NOTE: Dataset map will need to be configured to your own root directory.

Data formatting: 
- .txt files ONLY (.npy configurations to be added)
- Tab or space-delimited text files where column 0 is the independent variable (e.g., voltage/potential) and column 1 is the dependent variable (e.g., intensity/counts).
- The first 3 rows of each file are automatically skipped as headers (rows = 3). 
- This will work for .txt files saved directly from Gwyddion with OriginFriendly formatting. Please adjust if this is different. 
- Mask widths are hard coded, please change x_nm if needed. You will also need to change this manually in plotting function for x_lim.  
- Script assumes grain boundary and grain interior distributions fit a single gaussian. Change to mutli-gaussian (n_peaks = 2) if fits don't look right.
- Scipt assumes all values are input as volts and converts to mV for ease of plotting. 
- Plotting assumed for 600 DPI for output plots. 


Outputs: 
- Scatter plots for each individual image (errors taken from pcov, can change error bars to show spread by outputting FWHM instead). 

In [16]:
#Imports
import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from scipy.optimize import curve_fit
import pandas as pd
import scipy.stats as stats 
import matplotlib.colors as mcolors
import matplotlib.lines as mlines
import tkinter as tk
from tkinter import filedialog
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.colors import ListedColormap
import matplotlib.cm as cm
from matplotlib.ticker import MaxNLocator
from scipy.signal import find_peaks
from scipy.ndimage import gaussian_filter1d
from tqdm import tqdm

In [17]:
#Input directory and file saving function 
root = tk.Tk()
root.withdraw()
print("If pop up box not seen minimise spyder and select folder wanted for analysis")
#folder_path = filedialog.askdirectory(title='Select input folder')
#save_path = filedialog.askdirectory(title='Select Folder to Save Plot')

    
def save_plot_with_folder_dialog(fig, default_filename='plot.jpeg'):
    root = tk.Tk()
    root.withdraw()  # Hide the root window

    folder_path = filedialog.askdirectory(title='Select Folder to Save Plot')
    if not folder_path:
        print("Save cancelled.")
        return

    full_path = os.path.join(folder_path, default_filename)
    fig.savefig(full_path)
    print(f"Plot saved to: {full_path}")
    
def save_plot_to_folder(fig, folder_path, filename='plot.jpeg'):
    if not os.path.exists(folder_path):
        os.makedirs(folder_path)  # Create the folder if it doesn't exist
    
    full_path = os.path.join(folder_path, filename)
    fig.tight_layout()
    fig.savefig(full_path, dpi=600, bbox_inches='tight')
    print(f"Plot saved to: {full_path}")


If pop up box not seen minimise spyder and select folder wanted for analysis


In [18]:
#Colourmap formatting
# Change c1 and c2 based on the min and max colours to be chosen 

def hex_to_RGB(hex_str):
    
    """ #FFFFFF -> [255,255,255]"""
    
    return [int(hex_str[i:i+2], 16) for i in range(1,6,2)]

def get_color_gradient(c1, c2, n):

    assert n > 1
    c1_rgb = np.array(hex_to_RGB(c1))/255
    c2_rgb = np.array(hex_to_RGB(c2))/255
    mix_pcts = [x/(n-1) for x in range(n)]
    rgb_colors = [((1-mix)*c1_rgb + (mix*c2_rgb)) for mix in mix_pcts]
    return ["#" + "".join([format(int(round(val*255)), "02x") for val in item]) for item in rgb_colors]


colors=[]
c1='#1A85FF'#deep blue
c2='#D41159' # deep pink
c3= '#744db0' # purple

GB_U_color = "#5CBBFF"
GI_U_color = "#003357"
GB_P_color = "#D69AAB"
GI_P_color = "#8D1730"

In [ ]:
#----Configuring folders needed----
# number of rows in txt files to skip
rows = 3
DATASET_MAP = {
    'Unpass_1': {'parent': 'Unpassivated', 'type': 'Unpass1', 'colour1': "#5CBBFF", 'colour2': "#003357"},
    'Unpass_2': {'parent': 'Unpassivated', 'type': 'Unpass2', 'colour1': "#5CBBFF", 'colour2': "#003357"},
    'Unpass_3': {'parent': 'Unpassivated', 'type': 'Unpass3', 'colour1': "#5CBBFF", 'colour2': "#003357"},
    'Unpass_4': {'parent': 'Unpassivated', 'type': 'Unpass4', 'colour1': "#5CBBFF", 'colour2': "#003357"},
    'Pass_1': {'parent': 'Passivated', 'type': 'Pass1', 'colour1': "#D69AAB", 'colour2': "#8D1730"},
    'Pass_2': {'parent': 'Passivated', 'type': 'Pass2', 'colour1': "#D69AAB", 'colour2': "#8D1730"},
    'Pass_3': {'parent': 'Passivated', 'type': 'Pass3', 'colour1': "#D69AAB", 'colour2': "#8D1730"},
    'Pass_4': {'parent': 'Passivated', 'type': 'Pass4', 'colour1': "#D69AAB", 'colour2': "#8D1730"}
}

In [20]:
# --- Directory Selection Functions ---

def select_root_path():
    """Opens a single dialog to select the top-level ROOT_PATH."""
    root_path = filedialog.askdirectory(title='Select the Top-Level ROOT_PATH Folder')
    if not root_path:
        print("No root path selected. Exiting.")
        return None
    return root_path

In [21]:
#Single Gaussian Fitting

def gaussian(x, A, mu, sigma):
    return A * np.exp(-(x - mu)**2 / (2 * sigma**2))

In [22]:
def fit_gaussian_from_dataset(x_data, y_data):
    """
    Fit a Gaussian curve to the provided dataset (x_data, y_data).
    Args:
    - x_data: Array-like, x values of the data
    - y_data: Array-like, y values of the data
    
    Returns:
    - A_fit: Amplitude of the fitted Gaussian
    - mu_fit: Mean (center) of the fitted Gaussian
    - sigma_fit: Standard deviation (width) of the fitted Gaussian
    - popt: Optimal parameters from curve fitting
    - pcov: Covariance matrix
    """
    # Initial guess for the parameters [A, mu, sigma]
    initial_guess = [max(y_data), np.mean(x_data), np.std(x_data)]

    try:
        # Fit the Gaussian curve to the data
        popt, pcov = curve_fit(gaussian, x_data, y_data, p0=initial_guess, maxfev=100000000)
        
        # Extract fitted parameters
        A_fit, mu_fit, sigma_fit = popt
        # Print the fitted parameters for feedback
        #print(f"Fitted parameters:\nAmplitude: {A_fit}\nMean: {mu_fit}\nSigma: {sigma_fit}")
        #print(f"Raw data parameters are:\n Min = {min_raw}\n Mean = {mean_raw}\n Max = {max_raw}")
       
        # Return the fitted parameters and covariance matrix
        return A_fit, mu_fit, sigma_fit, popt, pcov

    except Exception as e:
        print(f"Error fitting Gaussian: {e}")
        return None, None, None, None, None

In [23]:
# --- Gaussian fitting---
def fit_gaussian_for_peaks (file_path, rows):
    """Performs Gaussian fitting on data from a given file path."""
    with open(file_path, 'r', encoding='utf-8', errors='replace') as f:
        data = np.loadtxt(f, skiprows=rows)
        
    x_data = data[:, 0]
    y_data = data[:, 1]
    
    # Estimate initial guess and peak count
    x_fit= np.linspace(np.min(x_data), np.max(x_data), 1000000)

    #For single gaussian: 
    A, mu, sigma, popt, pcov = fit_gaussian_from_dataset(x_data, y_data)
    y_fit=gaussian(x_fit,*popt)
    
    peak_index_max=np.argmax(y_fit)
    peak_centre=x_fit[peak_index_max]
    
    # Extract Gaussian parameters
    gaussians = []
    for i in range(len(popt) // 3):
        amp = popt[i*3]
        cen = popt[i*3+1]
        wid = abs(popt[i*3+2])
        gaussians.append({'amp': amp, 'cen': cen, 'wid': wid})

    highest = gaussians[np.argmax([g['amp'] for g in gaussians])]
    fwhm = 2.355 * highest['wid']
    # Fitting error calculation (for mu)
    peak_err = np.sqrt(pcov[1,1])
    return peak_centre, peak_err

In [24]:
#Specific grain boundary function
def grain_boundaries(GB_file_path, GI_file_path, pixel_index, save_path, GB_colour='input', GI_colour='input'):
    """
    Fits Gaussian to GB and GI data, plots the results, saves the plot, 
    and writes the analysis data to a text file.
    """
    # Pixel number corresponds to the 'pixels' value (1 to 5)
    pixels = pixel_index 
    #Plotting parameters
    dpi_setting = 600
    figsize_x, figsize_y = (2227/dpi_setting), (1498/dpi_setting)
    tick_length = 6 
    legend_font = 12
    scatter_size = 18
    base_font_size = 12
    line_width = 3
    # Perform fitting
    GB_peak_1, GB_err_1 = fit_gaussian_for_peaks(GB_file_path, rows)
    GI_peak_1, GI_err_1 = fit_gaussian_for_peaks(GI_file_path, rows)
    
    #Working out peak difference (GI-GB)
    peak_diff_1 = GI_peak_1 - GB_peak_1
    peak_diff_err_1 = np.sqrt(GB_err_1**2 + GI_err_1**2)
    params = [peak_diff_1, GB_peak_1, GI_peak_1, peak_diff_err_1, GB_err_1, GI_err_1]
    return params

In [25]:
def GB_diff_processing_final(root_path, dataset_key):
    """
    Processes all 5 pixel files for a given dataset using the fixed file structure:
    ROOT_PATH/parent/type/GB/1.txt
    """
    if not root_path:
        return [np.nan] * 5

    config = DATASET_MAP[dataset_key]
    parent_folder = config['parent']
    type_folder = config['type']
    colour1 = config['colour1']
    colour2 = config['colour2']

    print(f"\n--- Processing: {dataset_key} ---")

    # 1. Define the base folder containing GB/GI subfolders
    # This path is: ROOT_PATH/unpassivated/Dark_CPD
    base_data_folder = os.path.join(root_path, parent_folder, type_folder)
    
    # 2. Define the exact GB and GI folders
    GB_folder = os.path.join(base_data_folder, 'GB')
    GI_folder = os.path.join(base_data_folder, 'GI')
    
    # 3. Define the save path for intermediate plots and text files
    # This will be: ROOT_PATH/Plots/unpassivated/Dark_CPD
    save_path = os.path.join(root_path, 'Plots', parent_folder, type_folder)
    os.makedirs(save_path, exist_ok=True) # Create folder if it doesn't exist

    y_peaks = []
    GB_peaks = []
    GI_peaks = []
    y_err_peaks = []
    GB_err_peaks = []   
    GI_err_peaks = []
    
    # Check if the necessary folders exist before starting the loop
    if not os.path.isdir(GB_folder) or not os.path.isdir(GI_folder):
        print(f"Error: Could not find 'GB' and/or 'GI' folders at {base_data_folder}. Skipping.")
        return [np.nan] * 5

    # 4. Iterate through the 5 pixel files (1 to 5)
    for pixel_index in range(1, 6):
        
        # Construct the file paths: e.g., .../GB/1.txt and .../GI/1.txt
        file_name = f'{pixel_index}.txt'
        GB_file_path = os.path.join(GB_folder, file_name)
        GI_file_path = os.path.join(GI_folder, file_name)
        #print(GB_file_path)
        #print(GI_file_path)
        if os.path.exists(GB_file_path) and os.path.exists(GI_file_path):
            print(f"  Processing Pixel {pixel_index}...")
            params= grain_boundaries(
                GB_file_path, 
                GI_file_path, 
                pixel_index, 
                save_path, 
                colour1, 
                colour2
            )
            y_peaks.append(params[0])
            GB_peaks.append(params[1])
            GI_peaks.append(params[2])
            y_err_peaks.append(params[3])
            GB_err_peaks.append(params[4])
            GI_err_peaks.append(params[5])
        else:
            print(f" File(s) not found for Pixel {pixel_index}. Skipping.")
            print(f"{GB_file_path}")
            y_peaks.append(np.nan) 
            GB_peaks.append(np.nan)
            GI_peaks.append(np.nan)
            y_err_peaks.append(np.nan)
            GB_err_peaks.append(np.nan) 
            GI_err_peaks.append(np.nan)

    # Convert peak differences to mV and round
    y_peaks_adjusted = [round(x * 1000, 4) if not np.isnan(x) else np.nan for x in y_peaks]
    GB_peaks_adjusted = [round(x * 1000, 4) if not np.isnan(x) else np.nan for x in GB_peaks]
    GI_peaks_adjusted = [round(x * 1000, 4) if not np.isnan(x) else np.nan for x in GI_peaks]
    y_err_peaks_adjusted = [round(x * 1000, 4) if not np.isnan(x) else np.nan for x in y_err_peaks]
    GB_err_peaks_adjusted = [round(x * 1000, 4) if not np.isnan(x) else np.nan for x in GB_err_peaks]
    GI_err_peaks_adjusted = [round(x * 1000, 4) if not np.isnan(x) else np.nan for x in GI_err_peaks]
    full_params = [y_peaks_adjusted, GB_peaks_adjusted, GI_peaks_adjusted, y_err_peaks_adjusted, GB_err_peaks_adjusted, GI_err_peaks_adjusted]
    return full_params

In [ ]:
def plotting_individual_scatter(peaks, y_err, save_path, sample = "Unpassivated", save_name="plot.jpeg", scaletype="fixed"):
    x_nm = [20,60,100,140,180] # setting mask widths -- CHANGE IF MASK WIDTHS ARE DIFFERENT
    # Selecting scatter color based on sample type. 
    if sample == "Unpassivated":
        color = GI_U_color
    elif sample == "Passivated": 
        color = GI_P_color
    else: 
        print(input("Please enter a valid sample type: 'Unpassivated' or 'Passivated'"))

    #Set plot params for paper style figures: 
    errorbar_capsize = 4
    errorbar_linewidth = 1
    dpi_setting = 600
    figsize_x, figsize_y = (2227/dpi_setting), (1498/dpi_setting) 
    tick_length = 6 
    scatter_size = 4
    base_font_size = 12
    # Plotting
    plt.figure(figsize=(figsize_x, figsize_y))
    plt.rcParams.update({'font.size': base_font_size})
    ax = plt.gca()
    # Tick parameters for both axes
    ax.tick_params(direction='in', length=tick_length)
    ax.tick_params(axis='both')
    #Setting tick locations
    ax.xaxis.set_major_locator(MaxNLocator(nbins=5))
    ax.yaxis.set_major_locator(MaxNLocator(nbins=5))
    ax.set_xticks(x_nm)
    #Setting axis labels
    plt.xlabel("Mask width (nm)")
    plt.ylabel(r"$\Delta$CPD (mV)")
    ax.set_ylabel(r"$\Delta$CPD (mV)")
    # Adjustment for different scale types

    if scaletype == "fixed":
        y_lim_min, y_lim_max = map(int, input("Please enter the y-axis limits for fixed scaling in the format as integers: min,max").split(','))
        ax.set_ylim(y_lim_min, y_lim_max) # Takes input y lims.  
    elif scaletype == "own":
        ax.set_ylim(min(peaks)-max(y_err)-1, max(peaks)+max(y_err)+1) # adjusts y_axis for individal data
    
    ax.set_xlim(0, 200) # Change according to own mask widths if different from 20,60,100,140,180

    #Plotting
    plt.errorbar(x_nm, peaks, yerr = y_err, fmt = '-', capsize=errorbar_capsize, elinewidth=errorbar_linewidth, capthick =errorbar_linewidth, color=color, marker='o', markersize=scatter_size, ls = '')
    #plt.tight_layout()

    #saving
    #save_plot_to_folder(plt.gcf(), save_path, filename="{save_name}_{scaletype}.jpeg")
    plt.show() # If not showing or giving errors move to after return statement
    return


In [ ]:
# --- Main Execution Block ---
#Just writing data for scatter plots. Plotting in next block. 

ROOT_PATH = select_root_path()
FINAL_PLOTS_SAVE_PATH = filedialog.askdirectory(title='Select Folder to Save All Final Summary Plots')

if ROOT_PATH:
    # Get a list of all dataset keys
    all_datasets = list(DATASET_MAP.keys())
    
    # Initialize dictionary to store results
    all_results = {}
    
    print("\n--- Starting Batch Processing ---")
    
    # Loop through all datasets, calling the processing function
    for dataset_key in all_datasets:
        all_results[dataset_key] = GB_diff_processing_final(ROOT_PATH, dataset_key)

    # Assign results to original variables for summary plotting
    Unpass1, UnpassGB_1, UnpassGI_1, Unpass_err_peak_1, GB_err_peak_1, GI_err_peak_1 = all_results['Unpass_1']
    Unpass2, UnpassGB_2, UnpassGI_2, Unpass_err_peak_2, GB_err_peak_2, GI_err_peak_2 = all_results['Unpass_2']
    Unpass3, UnpassGB_3, UnpassGI_3, Unpass_err_peak_3, GB_err_peak_3, GI_err_peak_3 = all_results['Unpass_3']
    Unpass4, UnpassGB_4, UnpassGI_4, Unpass_err_peak_4, GB_err_peak_4, GI_err_peak_4 = all_results['Unpass_4']
    Pass1, PassGB_1, PassGI_1, Pass_err_peak_1, GB_err_peak_1, GI_err_peak_1 = all_results['Pass_1']
    Pass2, PassGB_2, PassGI_2, Pass_err_peak_2, GB_err_peak_2, GI_err_peak_2 = all_results['Pass_2']
    Pass3, PassGB_3, PassGI_3, Pass_err_peak_3, GB_err_peak_3, GI_err_peak_3 = all_results['Pass_3']
    Pass4, PassGB_4, PassGI_4, Pass_err_peak_4, GB_err_peak_4, GI_err_peak_4 = all_results['Pass_4']

In [ ]:
# Unpass1
plotting_individual_scatter(Unpass1, Unpass_err_peak_1, FINAL_PLOTS_SAVE_PATH, sample = "Unpassivated", save_name="Unpass1.jpeg", scaletype="fixed")
# Pass1
plotting_individual_scatter(Pass1, Pass_err_peak_1, FINAL_PLOTS_SAVE_PATH, sample = "Passivated", save_name="Pass1.jpeg", scaletype="fixed")
#Unpass2
plotting_individual_scatter(Unpass2, Unpass_err_peak_2, FINAL_PLOTS_SAVE_PATH, sample = "Unpassivated", save_name="Unpass2.jpeg", scaletype="fixed")
#Unpass3
plotting_individual_scatter(Unpass3, Unpass_err_peak_3, FINAL_PLOTS_SAVE_PATH, sample = "Unpassivated", save_name="Unpass3.jpeg", scaletype="fixed")
#Unpass4
plotting_individual_scatter(Unpass4, Unpass_err_peak_4, FINAL_PLOTS_SAVE_PATH, sample = "Unpassivated", save_name="Unpass4.jpeg", scaletype="fixed")
#Pass2
plotting_individual_scatter(Pass2, Pass_err_peak_2, FINAL_PLOTS_SAVE_PATH, sample = "Passivated", save_name="Pass2.jpeg", scaletype="fixed")
#Pass3
plotting_individual_scatter(Pass3, Pass_err_peak_3, FINAL_PLOTS_SAVE_PATH, sample = "Passivated", save_name="Pass3.jpeg", scaletype="fixed")
#Pass4
plotting_individual_scatter(Pass4, Pass_err_peak_4, FINAL_PLOTS_SAVE_PATH, sample = "Passivated", save_name="Pass4.jpeg", scaletype="fixed")